### TRIAL 3: Versión mejorada de generación de sCMBs sobre dataset CSIRO sujetos sanos

In [8]:
import os
import glob
import numpy as np
import nibabel as nib
import json
from scipy.ndimage import binary_erosion, generate_binary_structure, rotate
from skimage.transform import downscale_local_mean
import time
import shutil
from nilearn.image import resample_to_img

In [2]:
# ==========================================
# 1. CONFIGURACIÓN MASIVA (nnU-Net v2)
# ==========================================
# Ruta origen (AIBL limpio)
#INPUT_ROOT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/NoCMBSubject/data"
INPUT_ROOT_DIR = r"C:\Users\marta\Downloads\NoCMBSubject"

# Ruta destino: IMPORTANTE usar formato DatasetXXX_Nombre
DATASET_NAME = "Dataset113_SyntheticCMB"
#NNUNET_RAW_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/nnUNet_raw" 
NNUNET_RAW_DIR = r"C:\Users\marta\Downloads\nnUNet_raw" 
OUTPUT_DIR = os.path.join(NNUNET_RAW_DIR, DATASET_NAME)

# Parámetros
RANDOM_SEED = 42

# --- NUEVA LÓGICA DE DISTRIBUCIÓN (Trial 3) ---
TOTAL_SUBJECTS = 313
# 5% Limpios (), 50% con 8 sCMB (), 35% con 15 sCMB (), 10% con 20 sCMB ()
densities = [0]*15 + [10]*157 + [15]*110 + [20]*31 # Total: 3840 sCMBs

np.random.seed(RANDOM_SEED)
np.random.shuffle(densities) 

# Crear estructura de carpetas nnU-Net v2
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(os.path.join(OUTPUT_DIR, "imagesTr"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "labelsTr"), exist_ok=True)


In [9]:
# ==========================================
# 2. MOTOR MATEMÁTICO (Momeni Validado)
# ==========================================
def generate_momeni_gaussian(target_volume_mm3, voxel_size_mm, oversample=10):
    K = 1.175 
    term = (3 * target_volume_mm3) / (4 * np.pi * (K**3))
    sigma_t_mm = np.cbrt(term)
    
    # Aumentamos el rango de variación de los ejes para que no siempre sean casi-esferas
    rmin, rmax = 0.5, 1.2 # Subí de 0.4 a 0.5
    sigma_x_mm = sigma_t_mm * np.random.uniform(rmin, rmax)
    sigma_y_mm = sigma_t_mm * np.random.uniform(rmin, rmax)
    # sigma_z se calcula para mantener el volumen constante, lo cual es correcto
    sigma_z_mm = (sigma_t_mm**3) / (sigma_x_mm * sigma_y_mm)

    hr_vx_size = np.array(voxel_size_mm) / oversample
    sx_px = sigma_x_mm / hr_vx_size[0]
    sy_px = sigma_y_mm / hr_vx_size[1]
    sz_px = sigma_z_mm / hr_vx_size[2]
    
    max_sigma = max(sx_px, sy_px, sz_px)
    grid_size = int(max_sigma * 5) + 1 # Lo subo de 4 a 5
    if grid_size % 2 == 0: grid_size += 1
    
    cx, cy, cz = grid_size // 2, grid_size // 2, grid_size // 2
    
    x = np.arange(grid_size) - cx
    y = np.arange(grid_size) - cy
    z = np.arange(grid_size) - cz
    xx, yy, zz = np.meshgrid(x, y, z, indexing='ij')
    
    exponent = - ( (xx**2)/(2*sx_px**2) + (yy**2)/(2*sy_px**2) + (zz**2)/(2*sz_px**2) )
    gaussian = np.exp(exponent)

    # CAMBIO PARA EL TRIAL 3: QUITAMOS EL HALF-THRESHOLDING 

    # CAMBIO PARA EL TRIAL 3: AÑADIMOS VARIABILIDAD AL SHARPENING
    exponent_val = np.random.uniform(1.0, 2.0)
    gaussian = gaussian**exponent_val
    
    angle_x = np.random.uniform(-30, 30)
    angle_y = np.random.uniform(-30, 30)
    angle_z = np.random.uniform(-30, 30)
    
    img_rot = rotate(gaussian, angle_x, axes=(0,1), reshape=False, order=1)
    img_rot = rotate(img_rot, angle_y, axes=(1,2), reshape=False, order=1)
    img_rot = rotate(img_rot, angle_z, axes=(0,2), reshape=False, order=1)
    
    low_res_blob = downscale_local_mean(img_rot, (oversample, oversample, oversample))
    if low_res_blob.max() > 0:
        low_res_blob /= low_res_blob.max()
        
    return low_res_blob



def implant_and_label(image_data, label_data, center, volume_mm3, voxel_dims, strength):
    x, y, z = center
    lesion_pattern = generate_momeni_gaussian(volume_mm3, voxel_dims, oversample=10)
    
    p_shape = lesion_pattern.shape
    dx, dy, dz = p_shape[0]//2, p_shape[1]//2, p_shape[2]//2
    x_s, x_e = x - dx, x - dx + p_shape[0]
    y_s, y_e = y - dy, y - dy + p_shape[1]
    z_s, z_e = z - dz, z - dz + p_shape[2]
    
    if x_s < 0 or x_e > image_data.shape[0] or \
       y_s < 0 or y_e > image_data.shape[1] or \
       z_s < 0 or z_e > image_data.shape[2]:
        return image_data, label_data
    
    roi_img = image_data[x_s:x_e, y_s:y_e, z_s:z_e]
    if roi_img.shape != lesion_pattern.shape:
        lesion_pattern = lesion_pattern[:roi_img.shape[0], :roi_img.shape[1], :roi_img.shape[2]]
    
    # Cambios Trial 2:
    mask_multiplier = 1 - (lesion_pattern * strength)
    mask_multiplier = np.clip(mask_multiplier, 0.02, 1.0)

    image_data[x_s:x_e, y_s:y_e, z_s:z_e] = roi_img * mask_multiplier
    
    roi_label = label_data[x_s:x_e, y_s:y_e, z_s:z_e]
    lesion_mask_binary = (lesion_pattern >= 0.3).astype(int) # CAMBIO TRIAL3: antes era 0.5 y hacía que cogiéramos sólo núcleo duro
    label_data[x_s:x_e, y_s:y_e, z_s:z_e] = np.maximum(roi_label, lesion_mask_binary)
    # 

    return image_data, label_data

def get_segmentation_sampling_mask(image_path, img_shape):
    """
    Localiza la segmentación de SynthSeg y genera un mapa de probabilidad
    basado en pesos anatómicos validados (55% Cortex, 35% WM, 10% Deep GM).
    Incluir resampling  espacial dado que SynthSeg resamplea a vóxeles 1x1x1mm.
    """
    # 1. Localización corregida del archivo de segmentación
    # image_path: .../NoCMBSubject/data/archivo.nii.gz
    data_dir = os.path.dirname(image_path)          # .../NoCMBSubject/data
    root_dir = os.path.dirname(data_dir)          # .../NoCMBSubject
    
    filename = os.path.basename(image_path)
    seg_name = filename.replace('.nii.gz', '_segmentation.nii.gz')
    seg_path = os.path.join(root_dir, "segmentations", seg_name)

    if not os.path.exists(seg_path):
        raise FileNotFoundError(f"No se encuentra la segmentación en: {seg_path}")

    # 2. Carga de objetos NIfTI
    img_nii = nib.load(image_path)
    seg_nii = nib.load(seg_path)

    # 3. RESAMPLING ESPACIAL EXACTO
    # 'interpolation=nearest' asegura que las etiquetas no se degraden
    seg_resampled_nii = resample_to_img(seg_nii, img_nii, interpolation='nearest')
    seg_data = seg_resampled_nii.get_fdata().astype(int)

    # Ahora seg_data.shape es EXACTAMENTE img_shape (176, 256, 80)
    # y la anatomía coincide milímetro a milímetro.

    # 3. Definición de etiquetas SynthSeg
    labels_wm = [2, 41]
    labels_dgm = [10, 49, 11, 50, 12, 51, 13, 52, 17, 53, 18, 54]
    labels_cortex = list(range(1000, 3000))

    # 4. Creación del mapa de pesos
    # Inicializamos con ceros (probabilidad 0 para CSF, hueso y fondo)
    weight_map = np.zeros(img_shape, dtype=float)

    # Asignación de pesos objetivos (distribución relativa)
    # CAMBIO TRIAL 3: quitamos un 10% al cortex y se lo damos al DGM
    target_weights = {
        'cortex': 0.45,
        'wm': 0.35,
        'dgm': 0.20
    }

    # Calculamos las máscaras booleanas para cada grupo
    mask_cortex = np.isin(seg_data, labels_cortex)
    mask_wm = np.isin(seg_data, labels_wm)
    mask_dgm = np.isin(seg_data, labels_dgm)

    # Para que la probabilidad total de una región sea el target_weight, 
    # cada vóxel individual debe tener (peso_objetivo / número_de_vóxeles)
    for mask, weight in zip([mask_cortex, mask_wm, mask_dgm], target_weights.values()):
        n_voxels = np.sum(mask)
        if n_voxels > 0:
            weight_map[mask] = weight / n_voxels

    # 5. Obtener coordenadas y sus probabilidades
    valid_indices = np.argwhere(weight_map > 0)
    # Extraemos las probabilidades de esos índices específicos
    probs = weight_map[weight_map > 0]
    
    # Normalización final por seguridad (debe sumar 1)
    probs /= probs.sum()

    return valid_indices, probs


### Ejecutamos y dividimos en dataset train y test con muestreo estratificado

In [17]:
# ==========================================
# 3. PREPARACIÓN ESTRATIFICADA Y EJECUCIÓN
# ==========================================
print(f"Iniciando Pipeline Trial 3...")
nifti_files = sorted(glob.glob(os.path.join(INPUT_ROOT_DIR, "**/data/*.nii.gz"), recursive=True))

# Configuración de estratos (Total 313)
# Grupo: [Cantidad Total, Cantidad Test] -> pasamos 33 a test con la misma proporción
estratos = {
    0:  [15, 2],   # Sanos
    10: [157, 17], # Moderados
    15: [110, 12],   # Severos
    20: [31, 2] # Graves
}

# Crear listas de densidad para Train y Test
train_pool = [0]*13 + [10]*140 + [15]*98 + [20]*29
test_pool  = [0]*2  + [10]*17  + [15]*12 + [20]*2

np.random.seed(RANDOM_SEED)
np.random.shuffle(train_pool)
np.random.shuffle(test_pool)

# Unimos: los primeros 280 son Train, los últimos 33 son Test
final_densities = train_pool + test_pool
final_splits = ["Tr"] * 280 + ["Ts"] * 33

# Crear carpetas necesarias
for folder in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
    os.makedirs(os.path.join(OUTPUT_DIR, folder), exist_ok=True)

# --- BUCLE DE GENERACIÓN ---
for idx, file_path in enumerate(nifti_files):
    num_lesions = final_densities[idx]
    split = final_splits[idx]
    subject_id = f"CSIRO_{idx+1:03d}"
    
    print(f"[{idx+1}/313] {subject_id} | Objetivo: {num_lesions} sCMB | Destino: {split}")
    
    try:
        nii = nib.load(file_path)
        data_img = nii.get_fdata().astype(float)
        data_label = np.zeros(data_img.shape, dtype=np.uint8)
        voxel_dims = nii.header.get_zooms()

        if num_lesions > 0:
            valid_coords, probs = get_segmentation_sampling_mask(file_path, data_img.shape)
            if valid_coords is not None:
                random_indices = np.random.choice(len(valid_coords), size=num_lesions, replace=False, p=probs)
                for coord in valid_coords[random_indices]:
                    vol_rnd = np.random.triangular(1.2, 2.5, 15.0) # antes 0.8, 2.5, 15.0
                    str_rnd = np.random.triangular(0.70, 0.80, 0.95) # antes 0.70, 0.80, 0.98
                    data_img, data_label = implant_and_label(data_img, data_label, coord, vol_rnd, voxel_dims, str_rnd)

        # Guardado Directo
        img_ext = "_0000.nii.gz"
        nib.save(nib.Nifti1Image(data_img, nii.affine, nii.header), 
                 os.path.join(OUTPUT_DIR, f"images{split}", f"{subject_id}{img_ext}"))
        nib.save(nib.Nifti1Image(data_label, nii.affine, nii.header), 
                 os.path.join(OUTPUT_DIR, f"labels{split}", f"{subject_id}.nii.gz"))
        
    except Exception as e:
        print(f"    ERROR en {subject_id}: {e}")

# ==========================================
# 4. DATASET.JSON AUTOMÁTICO
# ==========================================
json_dict = {
    "channel_names": {"0": "SWI"},
    "labels": {"background": 0, "CMB": 1},
    "numTraining": 280,
    "file_ending": ".nii.gz",
    "name": DATASET_NAME,
    "reference": "Momeni Variant Method (Stratified Trial 3)",
    "description": "10% Healthy, 50% 10 sCMB, 35% 15 sCMB, 15% 20 sCMB. Integrated anatomical weights.",
    "overwrite_image_reader_writer": "SimpleITKIO"
}

with open(os.path.join(OUTPUT_DIR, "dataset.json"), 'w') as f:
    json.dump(json_dict, f, indent=4)

print(f"\nDATASET LISTO: {len(train_pool)} Entrenamiento / {len(test_pool)} Test")

Iniciando Pipeline Trial 3...
[1/313] CSIRO_001 | Objetivo: 10 sCMB | Destino: Tr


C:\Users\marta\AppData\Local\Temp\ipykernel_31416\2269584699.py:113: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  seg_resampled_nii = resample_to_img(seg_nii, img_nii, interpolation='nearest')
C:\Users\marta\AppData\Local\Temp\ipykernel_31416\2269584699.py:113: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  seg_resampled_nii = resample_to_img(seg_nii, img_nii, interpolation='nearest')


[2/313] CSIRO_002 | Objetivo: 10 sCMB | Destino: Tr


C:\Users\marta\AppData\Local\Temp\ipykernel_31416\2269584699.py:113: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  seg_resampled_nii = resample_to_img(seg_nii, img_nii, interpolation='nearest')
C:\Users\marta\AppData\Local\Temp\ipykernel_31416\2269584699.py:113: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  seg_resampled_nii = resample_to_img(seg_nii, img_nii, interpolation='nearest')


[3/313] CSIRO_003 | Objetivo: 15 sCMB | Destino: Tr


...

[312/313] CSIRO_312 | Objetivo: 10 sCMB | Destino: Ts


C:\Users\marta\AppData\Local\Temp\ipykernel_31416\2269584699.py:113: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  seg_resampled_nii = resample_to_img(seg_nii, img_nii, interpolation='nearest')
C:\Users\marta\AppData\Local\Temp\ipykernel_31416\2269584699.py:113: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  seg_resampled_nii = resample_to_img(seg_nii, img_nii, interpolation='nearest')


[313/313] CSIRO_313 | Objetivo: 15 sCMB | Destino: Ts


C:\Users\marta\AppData\Local\Temp\ipykernel_31416\2269584699.py:113: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  seg_resampled_nii = resample_to_img(seg_nii, img_nii, interpolation='nearest')
C:\Users\marta\AppData\Local\Temp\ipykernel_31416\2269584699.py:113: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  seg_resampled_nii = resample_to_img(seg_nii, img_nii, interpolation='nearest')



DATASET LISTO: 280 Entrenamiento / 33 Test


###  Comparativa ressolución imágenes originales con la segmentación de synthseg para estudiar el resampling

In [26]:
import os
import glob
import nibabel as nib
import pandas as pd

# --- CONFIGURACIÓN DE RUTAS ---
DATA_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/NoCMBSubject/data"
SEG_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/NoCMBSubject/segmentations"

results = []

nifti_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.nii.gz")))

print(f"Auditando {len(nifti_files)} pares de archivos...")

for f in nifti_files:
    fname = os.path.basename(f)
    sname = fname.replace('.nii.gz', '_segmentation.nii.gz')
    spath = os.path.join(SEG_DIR, sname)
    
    if not os.path.exists(spath):
        continue
        
    img_nii = nib.load(f)
    seg_nii = nib.load(spath)
    
    results.append({
        'Sujeto': fname,
        'SWI_Shape': img_nii.shape,
        'Seg_Shape': seg_nii.shape,
        'SWI_Voxel_mm': tuple(np.round(img_nii.header.get_zooms(), 2)),
        'Seg_Voxel_mm': tuple(np.round(seg_nii.header.get_zooms(), 2)),
        'Diff_Shape': img_nii.shape != seg_nii.shape
    })

df = pd.DataFrame(results)

# Resumen estadístico
diff_count = df['Diff_Shape'].sum()
print(f"\n--- RESULTADOS DEL ANÁLISIS ---")
print(f"Total analizados: {len(df)}")
print(f"Sujetos con discrepancia de dimensiones: {diff_count} ({diff_count/len(df)*100:.1f}%)")

# Mostrar los primeros 10 para inspección rápida
print("\nMuestra de los primeros 10 casos:")
print(df[['Sujeto', 'SWI_Shape', 'Seg_Shape', 'SWI_Voxel_mm', 'Seg_Voxel_mm']].head(10).to_string(index=False))

# Guardar a CSV para tu TFM
df.to_csv("/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/NoCMBSubject/auditoria_synthseg_resoluciones_trial2.csv", index=False)

Auditando 313 pares de archivos...

--- RESULTADOS DEL ANÁLISIS ---
Total analizados: 313
Sujetos con discrepancia de dimensiones: 313 (100.0%)

Muestra de los primeros 10 casos:
                           Sujeto      SWI_Shape       Seg_Shape       SWI_Voxel_mm    Seg_Voxel_mm
101_T1_MRI_SWI_BFC_50mm_HM.nii.gz (176, 256, 80) (165, 241, 140) (0.94, 0.94, 1.75) (1.0, 1.0, 1.0)
102_T1_MRI_SWI_BFC_50mm_HM.nii.gz (176, 256, 80) (165, 241, 140) (0.94, 0.94, 1.75) (1.0, 1.0, 1.0)
108_T1_MRI_SWI_BFC_50mm_HM.nii.gz (176, 256, 80) (165, 241, 140) (0.94, 0.94, 1.75) (1.0, 1.0, 1.0)
110_T1_MRI_SWI_BFC_50mm_HM.nii.gz (176, 256, 80) (166, 241, 140) (0.94, 0.94, 1.75) (1.0, 1.0, 1.0)
115_T1_MRI_SWI_BFC_50mm_HM.nii.gz (176, 256, 80) (165, 242, 141) (0.94, 0.94, 1.75) (1.0, 1.0, 1.0)
124_T1_MRI_SWI_BFC_50mm_HM.nii.gz (176, 256, 80) (165, 240, 140) (0.94, 0.94, 1.75) (1.0, 1.0, 1.0)
124_T2_MRI_SWI_BFC_50mm_HM.nii.gz (176, 256, 80) (165, 240, 141) (0.94, 0.94, 1.75) (1.0, 1.0, 1.0)
124_T3_MRI_SWI_BFC_50